<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/deep_agentpynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
import re
from typing import List, Dict, Any
from dataclasses import dataclass, asdict
from datetime import datetime

@dataclass
class Task:
    id: str
    description: str
    assigned_to: str = None
    status: str = "pending"
    result: Any = None
    dependencies: List[str] = None

    def __post_init__(self):
        if self.dependencies is None:
            self.dependencies = []

@dataclass
class Agent:
    name: str
    role: str
    expertise: str
    system_prompt: str

AGENT_REGISTRY = {
    "researcher": Agent(
        name="researcher",
        role="Research Specialist",
        expertise="Information gathering, analysis, and synthesis",
        system_prompt="You are a research specialist. Provide thorough research on topics."
    ),
    "medical_researcher": Agent(
        name="medical_researcher",
        role="Medical Research Expert",
        expertise="Deep medical research, specialist therapy, PubMed/Medline citations",
        system_prompt="You are a medical research expert. Focus on clinical evidence, specialized therapies, and provide formal bibliography citations from PubMed and Medline."
    ),
    "coder": Agent(
        name="coder",
        role="Software Engineer",
        expertise="Writing clean, efficient code with best practices",
        system_prompt="You are an expert programmer. Write clean, well-documented code."
    ),
    "writer": Agent(
        name="writer",
        role="Content Writer",
        expertise="Clear communication and documentation",
        system_prompt="You are a professional writer. Create clear, engaging content."
    ),
    "analyst": Agent(
        name="analyst",
        role="Data Analyst",
        expertise="Data interpretation and insights",
        system_prompt="You are a data analyst. Provide clear insights from data."
    )
}

class LocalLLM:
    def __init__(self, model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        ) if torch.cuda.is_available() else None
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate(self, prompt: str, max_tokens: int = 300) -> str:
        formatted_prompt = f"<|system|>\nYou are a helpful AI assistant.</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"
        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
            padding=True
        )
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True
            )
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "<|assistant|>" in full_response:
            return full_response.split("<|assistant|>")[-1].strip()
        return full_response[len(formatted_prompt):].strip()

class ManagerAgent:
    def __init__(self, model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        self.llm = LocalLLM(model_name)
        self.agents = AGENT_REGISTRY
        self.tasks: Dict[str, Task] = {}
        self.execution_log = []

    def log(self, message: str):
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_entry = f"[{timestamp}] {message}"
        self.execution_log.append(log_entry)
        print(log_entry)

    def decompose_goal(self, goal: str) -> List[Task]:
        self.log(f"‐ Decomposing goal: {goal}")
        agent_info = "\n".join([f"- {name}: {agent.expertise}" for name, agent in self.agents.items()])
        prompt = f"""Break down this goal into 3 specific subtasks. Assign each to the best agent.\n\nGoal: {goal}\n\nAvailable agents:\n{agent_info}\n\nRespond ONLY with a JSON array."""
        response = self.llm.generate(prompt, max_tokens=250)
        try:
            json_match = re.search(r'\[\s*\{.*?\}\s*\]', response, re.DOTALL)
            if json_match:
                tasks_data = json.loads(json_match.group())
            else:
                raise ValueError("No JSON found")
        except:
            tasks_data = self._create_default_tasks(goal)

        tasks = []
        for i, task_data in enumerate(tasks_data[:3]):
            task = Task(
                id=task_data.get('id', f'task_{i+1}'),
                description=task_data.get('description', f'Work on: {goal}'),
                assigned_to=task_data.get('assigned_to', list(self.agents.keys())[i % len(self.agents)]),
                dependencies=task_data.get('dependencies', [] if i == 0 else [f'task_{i}'])
            )
            self.tasks[task.id] = task
            tasks.append(task)
            self.log(f"  ✓ {task.id}: {task.description[:50]}... → {task.assigned_to}")

        return tasks

    def _create_default_tasks(self, goal: str) -> List[Dict]:
        if any(word in goal.lower() for word in ['medicin', 'therap', 'health', 'clinic']):
            return [
                {"id": "task_1", "description": f"Deep medical research on: {goal} with PubMed sources", "assigned_to": "medical_researcher", "dependencies": []},
                {"id": "task_2", "description": f"Analyze clinical findings", "assigned_to": "analyst", "dependencies": ["task_1"]},
                {"id": "task_3", "description": f"Write specialist medical summary", "assigned_to": "writer", "dependencies": ["task_2"]}
            ]
        if any(word in goal.lower() for word in ['code', 'program', 'implement', 'algorithm']):
            return [
                {"id": "task_1", "description": f"Research and explain the concept: {goal}", "assigned_to": "researcher", "dependencies": []},
                {"id": "task_2", "description": f"Write code implementation for: {goal}", "assigned_to": "coder", "dependencies": ["task_1"]},
                {"id": "task_3", "description": f"Create documentation and examples", "assigned_to": "writer", "dependencies": ["task_2"]}
            ]
        return [
            {"id": "task_1", "description": f"Research: {goal}", "assigned_to": "researcher", "dependencies": []},
            {"id": "task_2", "description": f"Analyze findings and structure content", "assigned_to": "analyst", "dependencies": ["task_1"]},
            {"id": "task_3", "description": f"Write comprehensive response", "assigned_to": "writer", "dependencies": ["task_2"]}
        ]

    def execute_task(self, task: Task, context: Dict[str, Any] = None) -> str:
        self.log(f"‐ Executing {task.id} with {task.assigned_to}")
        task.status = "in_progress"
        agent = self.agents[task.assigned_to]
        context_str = ""
        if context and task.dependencies:
            context_str = "\n\nContext from previous tasks:\n"
            for dep_id in task.dependencies:
                if dep_id in context:
                    context_str += f"- {context[dep_id][:150]}...\n"

        prompt = f"""{agent.system_prompt}\n\nTask: {task.description}{context_str}\n\nProvide a clear, concise response:"""
        result = self.llm.generate(prompt, max_tokens=250)
        task.result = result
        task.status = "completed"
        self.log(f"  ✓ Completed {task.id}")
        return result

    def synthesize_results(self, goal: str, results: Dict[str, str]) -> str:
        self.log("‐ Synthesizing final results")
        results_text = "\n\n".join([f"Task {tid}:\n{res[:200]}" for tid, res in results.items()])
        prompt = f"""Combine these task results into one final coherent answer.\n\nOriginal Goal: {goal}\n\nTask Results:\n{results_text}\n\nFinal comprehensive answer:"""
        return self.llm.generate(prompt, max_tokens=350)

    def execute_goal(self, goal: str) -> Dict[str, Any]:
        self.log(f"\n{'='*60}\n‐ Starting Manager Agent\n{'='*60}")
        tasks = self.decompose_goal(goal)
        results = {}
        completed = set()
        max_iterations = len(tasks) * 2
        iteration = 0

        while len(completed) < len(tasks) and iteration < max_iterations:
            iteration += 1
            for task in tasks:
                if task.id in completed:
                    continue
                deps_met = all(dep in completed for dep in task.dependencies)
                if deps_met:
                    result = self.execute_task(task, results)
                    results[task.id] = result
                    completed.add(task.id)

        final_output = self.synthesize_results(goal, results)
        self.log(f"\n{'='*60}\n✓ Execution Complete!\n{'='*60}\n")

        return {
            "goal": goal,
            "tasks": [asdict(task) for task in tasks],
            "final_output": final_output,
            "execution_log": self.execution_log
        }

def demo_basic():
    manager = ManagerAgent()
    goal = "Explain binary search algorithm with a simple example"
    result = manager.execute_goal(goal)
    print("\n" + "="*60)
    print("FINAL OUTPUT")
    print("="*60)
    print(result["final_output"])
    return result

def demo_coding():
    manager = ManagerAgent()
    goal = "Implement a function to find the maximum element in a list"
    result = manager.execute_goal(goal)
    print("\n" + "="*60)
    print("FINAL OUTPUT")
    print("="*60)
    print(result["final_output"])
    return result

def demo_custom(custom_goal: str):
    manager = ManagerAgent()
    result = manager.execute_goal(custom_goal)
    print("\n" + "="*60)
    print("FINAL OUTPUT")
    print("="*60)
    print(result["final_output"])
    return result

if __name__ == "__main__":
    print("‐ Manager Agent Tutorial - APIless Local Version")
    print("="*60)
    print("Using TinyLlama (1.1B) - Fast & efficient!\n")
    result = demo_basic()
    print("\n\n‐ Try more:")
    print("  - demo_coding()")
    print("  - demo_custom('your goal here')")

‐ Manager Agent Tutorial - APIless Local Version
Using TinyLlama (1.1B) - Fast & efficient!



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[18:59:44] 
‐ Starting Manager Agent
[18:59:44] ‐ Decomposing goal: Explain binary search algorithm with a simple example
[19:01:40]   ✓ task_1: Research and explain the concept: Explain binary s... → researcher
[19:01:40]   ✓ task_2: Write code implementation for: Explain binary sear... → coder
[19:01:40]   ✓ task_3: Create documentation and examples... → writer
[19:01:40] ‐ Executing task_1 with researcher
[19:03:37]   ✓ Completed task_1
[19:03:37] ‐ Executing task_2 with coder
[19:05:28]   ✓ Completed task_2
[19:05:28] ‐ Executing task_3 with writer
[19:07:13]   ✓ Completed task_3
[19:07:13] ‐ Synthesizing final results
[19:09:59] 
✓ Execution Complete!


FINAL OUTPUT
Based on the task results, here is a final comprehensive answer:

Task 1: Explain binary search algorithm with a simple example

Task 2: A detailed research on the concept of binary search algorithm with a simple example

Task 3: A simple example for binary search with clear and concise documentation and examples

Bina

### 🚀 Prova il tuo Manager Agent
Inserisci un obiettivo qui sotto (es. 'Spiega come funziona il ciclo dell'acqua' oppure 'Scrivi una funzione Python per invertire una stringa') e premi il tasto Play a sinistra della cella.

In [ ]:
obiettivo_personalizzato = """Analisi clinica neonato: genitali ambigui, cariotipo 46,XX. Lab: 17-OH-progesterone elevato, cortisolo basso, aldosterone basso, kaliemia alta. Sospetto diagnostico tra: Deficit 11beta-idrossilasi, Sindrome di Turner, Sindrome di Rokitansky, Deficit 21-idrossilasi. Fornisci spiegazione e riferimenti bibliografici."""

# Esegui il manager
risultato = demo_custom(obiettivo_personalizzato)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[19:15:49] 
‐ Starting Manager Agent
[19:15:49] ‐ Decomposing goal: Analisi clinica neonato: genitali ambigui, cariotipo 46,XX. Lab: 17-OH-progesterone elevato, cortisolo basso, aldosterone basso, kaliemia alta. Sospetto diagnostico tra: Deficit 11beta-idrossilasi, Sindrome di Turner, Sindrome di Rokitansky, Deficit 21-idrossilasi. Fornisci spiegazione e riferimenti bibliografici.
[19:18:09]   ✓ task_1: Deep medical research on: Analisi clinica neonato:... → medical_researcher
[19:18:09]   ✓ task_2: Analyze clinical findings... → analyst
[19:18:09]   ✓ task_3: Write specialist medical summary... → writer
[19:18:09] ‐ Executing task_1 with medical_researcher


KeyboardInterrupt: 

In [ ]:
import gc
import torch

# Pulizia memoria GPU/RAM se necessario
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Re-inizializzazione e nuova esecuzione del caso clinico
manager_medico = ManagerAgent()

obiettivo_medico = """Analisi clinica neonato: genitali ambigui, cariotipo 46,XX.
Lab: 17-OH-progesterone elevato, cortisolo basso, aldosterone basso, kaliemia alta.
Sospetto diagnostico: Deficit 21-idrossilasi.
Fornisci spiegazione fisiopatologica e riferimenti bibliografici da PubMed."""

risultato_medico = manager_medico.execute_goal(obiettivo_medico)

print("\n" + "="*60)
print("RISULTATO ANALISI MEDICA")
print("="*60)
print(risultato_medico['final_output'])


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[19:20:18] 
‐ Starting Manager Agent
[19:20:18] ‐ Decomposing goal: Analisi clinica neonato: genitali ambigui, cariotipo 46,XX. 
Lab: 17-OH-progesterone elevato, cortisolo basso, aldosterone basso, kaliemia alta. 
Sospetto diagnostico: Deficit 21-idrossilasi. 
Fornisci spiegazione fisiopatologica e riferimenti bibliografici da PubMed.
[19:22:35]   ✓ task_1: Deep medical research on: Analisi clinica neonato:... → medical_researcher
[19:22:35]   ✓ task_2: Analyze clinical findings... → analyst
[19:22:35]   ✓ task_3: Write specialist medical summary... → writer
[19:22:35] ‐ Executing task_1 with medical_researcher
[19:24:40]   ✓ Completed task_1
[19:24:40] ‐ Executing task_2 with analyst
[19:26:12]   ✓ Completed task_2
[19:26:12] ‐ Executing task_3 with writer
[19:27:58]   ✓ Completed task_3
[19:27:58] ‐ Synthesizing final results
[19:30:01] 
✓ Execution Complete!


RISULTATO ANALISI MEDICA
Analyze the clinical evidence presented by the neonate with ambiguous genitalia, including the pres

In [ ]:
# Analisi del nuovo caso clinico fornito dall'utente
obiettivo_diagnostico = """Caso Clinico: Neonato 46,XX con genitali ambigui.
Lab: 17-OH-progesterone ELEVATO, cortisolo BASSO, aldosterone BASSO/LIMITI INF, kaliemia ALTA.
Valuta tra: Deficit 11beta-idrossilasi, Sindrome di Turner, Sindrome di Rokitansky, Deficit 21-idrossilasi.
Fornisci la diagnosi corretta, la spiegazione fisiopatologica e i riferimenti bibliografici."""

# Esecuzione del Manager Agent
risultato_diagnosi = manager_medico.execute_goal(obiettivo_diagnostico)

print("\n" + "="*60)
print("VERIFICA DIAGNOSTICA")
print("="*60)
print(risultato_diagnosi['final_output'])


[20:04:19] 
‐ Starting Manager Agent
[20:04:19] ‐ Decomposing goal: Caso Clinico: Neonato 46,XX con genitali ambigui.
Lab: 17-OH-progesterone ELEVATO, cortisolo BASSO, aldosterone BASSO/LIMITI INF, kaliemia ALTA.
Valuta tra: Deficit 11beta-idrossilasi, Sindrome di Turner, Sindrome di Rokitansky, Deficit 21-idrossilasi.
Fornisci la diagnosi corretta, la spiegazione fisiopatologica e i riferimenti bibliografici.
[20:06:09]   ✓ task_1: Work on: Caso Clinico: Neonato 46,XX con genitali ... → researcher
[20:06:09]   ✓ task_2: Work on: Caso Clinico: Neonato 46,XX con genitali ... → medical_researcher
[20:06:09]   ✓ task_3: Work on: Caso Clinico: Neonato 46,XX con genitali ... → coder
[20:06:09] ‐ Executing task_1 with researcher
[20:08:19]   ✓ Completed task_1
[20:08:19] ‐ Executing task_2 with medical_researcher
[20:10:50]   ✓ Completed task_2
[20:10:50] ‐ Executing task_3 with coder
[20:13:16]   ✓ Completed task_3
[20:13:16] ‐ Synthesizing final results


KeyboardInterrupt: 

In [ ]:
# Calcolo dell'accuratezza del test
veri_positivi = 190
totale_positivi_reali = 200
veri_negativi = 760
totale_negativi_reali = 800

totale_donne = 1000
previsioni_corrette = veri_positivi + veri_negativi

accuratezza = previsioni_corrette / totale_donne

print(f"Totale casi corretti: {previsioni_corrette} su {totale_donne}")
print(f"Accuratezza calcolata: {accuratezza}")

Totale casi corretti: 950 su 1000
Accuratezza calcolata: 0.95


In [ ]:
obiettivo_clinico_finale = """Analisi Differenziale Caso Clinico:
- Paziente: Neonato 46,XX con genitali ambigui.
- Laboratorio: 17-OHP elevato, Cortisolo basso, Aldosterone basso, Potassio (K+) alto.
- Opzioni: Deficit 11beta-idrossilasi, Sindrome di Turner, Sindrome di Rokitansky, Deficit 21-idrossilasi.

Identifica la patologia corretta, spiega il meccanismo biochimico del blocco enzimatico e perché si verifica l'iperkaliemia."""

# Utilizzo del manager già inizializzato
risultato_clinico = manager_medico.execute_goal(obiettivo_clinico_finale)

print("\n" + "="*60)
print("DIAGNOSI MEDICA E FISIOPATOLOGIA")
print("="*60)
print(risultato_clinico['final_output'])


[20:18:23] 
‐ Starting Manager Agent
[20:18:23] ‐ Decomposing goal: Analisi Differenziale Caso Clinico:
- Paziente: Neonato 46,XX con genitali ambigui.
- Laboratorio: 17-OHP elevato, Cortisolo basso, Aldosterone basso, Potassio (K+) alto.
- Opzioni: Deficit 11beta-idrossilasi, Sindrome di Turner, Sindrome di Rokitansky, Deficit 21-idrossilasi.

Identifica la patologia corretta, spiega il meccanismo biochimico del blocco enzimatico e perché si verifica l'iperkaliemia.
[20:20:53]   ✓ task_1: Deep medical research on: Analisi Differenziale Ca... → medical_researcher
[20:20:53]   ✓ task_2: Analyze clinical findings... → analyst
[20:20:53]   ✓ task_3: Write specialist medical summary... → writer
[20:20:53] ‐ Executing task_1 with medical_researcher
[20:23:10]   ✓ Completed task_1
[20:23:10] ‐ Executing task_2 with analyst
[20:25:00]   ✓ Completed task_2
[20:25:00] ‐ Executing task_3 with writer
[20:26:43]   ✓ Completed task_3
[20:26:43] ‐ Synthesizing final results
[20:29:01] 
✓ Execution C

In [ ]:
obiettivo_diagnostico_nuovo = """Analisi Differenziale Caso Clinico:
- Paziente: Neonato 46,XX con genitali ambigui.
- Laboratorio: 17-OH-progesterone elevato, cortisolo basso, aldosterone basso, kaliemia alta.
- Opzioni: Iperpalsia surrenalica congenita causata da deficit di 11beta-idrossilasi, Sindrome di Turner, Nessuna delle precedenti, Sindrome di Rokitansky, Iperpalsia surrenalica congenita causata da deficit di 21-idrossilasi.

Identifica la patologia corretta tra quelle proposte, spiega il meccanismo fisiopatologico e l'iperkaliemia."""

risultato_diagnostico_nuovo = manager_medico.execute_goal(obiettivo_diagnostico_nuovo)

print("\n" + "="*60)
print("DIAGNOSI E SPIEGAZIONE FISIOPATOLOGICA")
print("="*60)
print(risultato_diagnostico_nuovo['final_output'])

[20:35:44] 
‐ Starting Manager Agent
[20:35:44] ‐ Decomposing goal: Analisi Differenziale Caso Clinico:
- Paziente: Neonato 46,XX con genitali ambigui.
- Laboratorio: 17-OH-progesterone elevato, cortisolo basso, aldosterone basso, kaliemia alta.
- Opzioni: Iperpalsia surrenalica congenita causata da deficit di 11beta-idrossilasi, Sindrome di Turner, Nessuna delle precedenti, Sindrome di Rokitansky, Iperpalsia surrenalica congenita causata da deficit di 21-idrossilasi.

Identifica la patologia corretta tra quelle proposte, spiega il meccanismo fisiopatologico e l'iperkaliemia.
